# Client-Centric Consistency in a Replicated MongoDB Cluster

**Course project: distributed database with tunable consistency**
Group members: _[names here]_ · Date: September 2026

---

## What this notebook does

This notebook is the **complete, reproducible experiment** for the project. Run it top to bottom and it will:

1. Build a **3-node MongoDB 7.0 replica set** as Docker containers (no manual terminal work except a one-time `/etc/hosts` edit).
2. Record the software versions and architecture for the report.
3. State our **predictions** (hypotheses) for whether each of the four client-centric consistency models holds under four client configurations.
4. Run **repeated, randomised trials** of each model × configuration under seven scenarios:
   normal operation, induced replication lag, secondary failure, primary failure, loss of majority, partition of a secondary, partition of the primary.
5. Tabulate observed violation rates against predictions and export the results as CSV for the report.

Every section explains **what** is being done and **why**, so the notebook doubles as the experimental-design chapter of the report.

## How to reproduce (README)

| Step | Command / action |
|---|---|
| 1 | Install Docker Desktop and make sure it is running. |
| 2 | One-time: add `127.0.0.1 mongo1`, `127.0.0.1 mongo2`, `127.0.0.1 mongo3` to `/etc/hosts` (`sudo nano /etc/hosts`). Reason: replica-set members advertise themselves by container name; the notebook runs *outside* Docker and must resolve those names to `localhost`. |
| 3 | `pip install pymongo pandas` into the Jupyter kernel's Python (cell 1 below does this automatically). |
| 4 | Run all cells in order (Kernel → Restart & Run All). Full run takes roughly 15–20 minutes because several scenarios deliberately wait for elections and timeouts. |
| 5 | Results are printed inline and saved to `consistency_results.csv`. |

Ports used on the host: 27017, 27018, 27019. Container names: `mongo1`, `mongo2`, `mongo3`. Docker networks: `zz-repl` (replication) and `net-mongoN` (one per node, carries the published port). Containers are started with `--cap-add NET_ADMIN` and get `iptables` installed so that S5/S6 can build a real partition.

In [1]:
# Cell 1 — install the two Python libraries this notebook needs, into THIS kernel's Python.
# (`sys.executable` guarantees we install into the interpreter Jupyter is actually running,
#  which is not always the same one that `pip` on the terminal points to.)
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymongo", "pandas"], check=True)
print("pymongo and pandas are installed for:", sys.executable)

pymongo and pandas are installed for: /usr/local/bin/python3



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip3.13 install --upgrade pip


## 1. Background: the four client-centric consistency models

Client-centric consistency (Tanenbaum & van Steen, *Distributed Systems*, ch. 7) describes guarantees given to a **single client (session)** even when the replicas as a whole are only eventually consistent. With one client that reads and writes a data item `x`, possibly at different replicas:

| Model | Guarantee | Example of a violation |
|---|---|---|
| **Read-your-writes (RYW)** | After the client writes `x`, every later read of `x` by that client returns that value or a newer one. | You post a comment, refresh the page, and your comment is gone (the refresh hit a replica that had not received the write). |
| **Monotonic reads (MR)** | Once the client has read a value of `x`, later reads never return an *older* value. | You see 5 unread e-mails, then reload and see 3 unread (an older state). |
| **Monotonic writes (MW)** | Two writes by the same client are applied to every replica in the order they were issued. | You set your name to "A" then "B"; a replica applies "B" then "A" and ends up showing "A". |
| **Writes-follow-reads (WFR)** | A write that follows a read of `x` is applied, at every replica, *after* the value that was read. | Someone replies to a comment; a replica shows the reply but not the comment it replies to. |

## 2. MongoDB's consistency knobs and how they map to these models

MongoDB replica sets have a **single primary** that accepts all writes and replicates them, in a totally ordered operation log (oplog), to **secondaries**. The client controls consistency with four settings:

| Knob | Values we use | Effect |
|---|---|---|
| **Write concern `w`** | `1`, `"majority"` | How many replicas must acknowledge a write before it is reported successful. `w:1` = primary only (fast, but a secondary may lag); `w:"majority"` = a majority has the write (durable across failover). |
| **Read concern** | `"local"`, `"majority"` | `local` returns the node's latest data even if not yet majority-committed; `majority` returns only majority-committed data (never rolled back). Neither, on its own, says anything about *which* node you read from. |
| **Read preference** | `primary`, `secondary` (tagged to a specific node) | Which member serves reads. Reading from a secondary is where staleness comes from. |
| **Causally consistent session** | off / on | The driver attaches the cluster time of the session's last operation to each read (`afterClusterTime`); a secondary **waits** until it has applied up to that time before answering. MongoDB documents that a causal session with `w:"majority"` + `readConcern:"majority"` guarantees exactly the four models above. |

### Client configurations under test

| Config | Write concern | Read concern | Reads served by | Causal session | Intuition |
|---|---|---|---|---|---|
| **C1 weak** | `w:1` | `local` | secondary (`mongo3`) | no | "Fast and loose": nothing stops a stale read. |
| **C2 majority** | `w:"majority"` | `majority` | secondary (`mongo3`) | no | Stronger *durability*, but still no session tracking: a lagging secondary still answers with old (majority-committed) data. |
| **C3 causal** | `w:"majority"` | `majority` | secondary (`mongo3`) | **yes** | The configuration MongoDB documents as providing all four models. |
| **C4 primary** | `w:1` | `local` | primary | no | Avoid staleness by never reading a replica. |

## 3. Predictions (hypotheses stated before running anything)

Reasoning from the mechanism above:

* **RYW and MR** depend on whether the read can land on a replica that has not yet applied the write. With C1 and C2 nothing prevents that, so they are **not guaranteed** and should fail whenever the secondary lags (partition, delayed member). C3 forces the secondary to catch up first, and C4 never reads a replica, so both **hold**.
* **MW** is guaranteed by architecture: one primary, one totally ordered oplog, secondaries apply ops in order. **Holds** in all configs. (The only theoretical exception is an un-replicated `w:1` write being rolled back after a primary crash, which we discuss but do not test.)
* **WFR** likewise holds structurally: if client A read `x=v` (from any node, which only serves values already in the oplog) and then wrote `y`, then `x=v` precedes `y` in the oplog and every node applies them in that order. **Holds** in all configs.

| Model | C1 weak | C2 majority | C3 causal | C4 primary |
|---|:-:|:-:|:-:|:-:|
| Read-your-writes | ✗ may fail | ✗ may fail | ✓ holds | ✓ holds |
| Monotonic reads | ✗ may fail | ✗ may fail | ✓ holds | ✓ holds |
| Monotonic writes | ✓ holds | ✓ holds | ✓ holds | ✓ holds |
| Writes-follow-reads | ✓ holds | ✓ holds | ✓ holds | ✓ holds |

An important methodological point: "may fail" means *not guaranteed*. On a single laptop where replication takes a few milliseconds, C1/C2 will usually *appear* to work. A prediction of "may fail" is only **confirmed** if we observe at least one violation, and it is only **meaningfully tested** if we create conditions (replication lag, partition) where a violation is possible. That is why Scenarios S1, S5 and S6 exist.

We also expect a **latency cost** for C3: its reads block until the secondary catches up, so we record the mean time per trial.

## 4. Deployment architecture

```
             host (macOS) — this notebook, pymongo
                 │ localhost:27017      │ localhost:27018      │ localhost:27019
            net-mongo1             net-mongo2             net-mongo3      <- one private network per node (host access)
   ┌─────────────┴───────┐  ┌───────────┴─────────┐  ┌─────────┴───────────┐
   │ container mongo1    │  │ container mongo2    │  │ container mongo3    │
   │ mongod --port 27017 │  │ mongod --port 27018 │  │ mongod --port 27019 │
   │ priority 3          │  │ priority 2          │  │ priority 1          │
   └──────────┬──────────┘  └──────────┬──────────┘  └──────────┬──────────┘
              └────────────── docker network "zz-repl" ───────────────┘   <- replication traffic
```

Design decisions, and why:

* **Each container listens on a different port that is identical inside and outside Docker** (`--port 27018` and `-p 27018:27018`). A replica-set member advertises one `host:port` to everyone; that address must work both for the other containers (via Docker DNS) and for the notebook on the host (via `/etc/hosts`). Using the default 27017 everywhere internally would make `mongo2:27017` unreachable from the host.
* **Member priorities 3 / 2 / 1** make elections deterministic: `mongo1` is primary whenever it is healthy, `mongo2` takes over if `mongo1` fails, and `mongo3` is always a secondary unless both others are down. This lets every experiment read from a **known** secondary (`mongo3`) and makes results reproducible.
* **Member tags** (`{name: "mongoN"}`) let the driver target a specific secondary with a read preference, so we control exactly which replica answers each read.
* **`electionTimeoutMillis: 5000`** (default 10000) shortens failover so the failure scenarios run faster; it does not change any consistency semantics.
* **Two networks per container.** Each container is created on its own private network (`net-mongoN`), which is where Docker attaches the published host port, and is then also joined to the shared `zz-repl` network that carries replication traffic. The replication network is named so that it sorts *after* the private networks: in our first run, with the replication network named `mongo-cluster`, `docker network disconnect` made the isolated node unreachable from the host as well (every S5/S6 trial failed with `ServerSelectionTimeoutError`), so the "partition" was indistinguishable from a crash. Our working hypothesis is that Docker chose the alphabetically-first network as the container's default gateway. We did not rely on that hypothesis: partitions are now built with **`iptables` rules inside the container** (dropping traffic to/from the other members' replication-network IPs), which leaves the host's published port untouched by construction. `docker network disconnect` remains as a fallback, and every partition scenario first *verifies* that the node is still reachable from the host and has lost sight of the primary before any trial is counted.
* **Named Docker volumes** for data, because bind-mounting a macOS folder into the MongoDB image causes permission failures on restart (we hit this during setup).

In [2]:
# Cell 2 — imports and global constants. Everything configurable lives here.
import sys, subprocess, time, uuid, json, statistics, platform
from datetime import datetime
import pandas as pd
import pymongo
from pymongo import MongoClient, WriteConcern
from pymongo.read_concern import ReadConcern
from pymongo.read_preferences import Primary, Secondary, SecondaryPreferred
from pymongo.errors import PyMongoError

# ---- cluster definition ---------------------------------------------------
RS_NAME   = "rs0"
NETWORK   = "zz-repl"        # name sorts AFTER "net-mongoN" on purpose, see Section 4
IMAGE     = "mongo:7.0"
NODES     = {"mongo1": 27017, "mongo2": 27018, "mongo3": 27019}   # name -> port (same inside & outside Docker)
PRIORITY  = {"mongo1": 3, "mongo2": 2, "mongo3": 1}                # deterministic election order
TARGET_SECONDARY   = "mongo3"                                       # the replica all "secondary" reads go to
ELECTION_TIMEOUT_MS = 5000
HEARTBEAT_MS         = 1000    # driver re-scans the replica set this often (default is 10000ms,
                                # far slower than the reconfigs/failovers we trigger below)

# ---- experiment knobs -----------------------------------------------------
N_TRIALS      = 20     # trials per (model, config) in fast scenarios
N_TRIALS_SLOW = 8      # trials in scenarios where causal reads must wait for a lagging node
MAX_TIME_MS   = 6000   # upper bound a single read may block (causal reads wait for the secondary)
W_TIMEOUT_MS  = 5000   # upper bound a write may wait for acknowledgements
DELAY_SECS    = 3      # replication delay injected in Scenario S1
OBSERVE_SECS  = 1.0    # how long MW/WFR trials watch a node for the write to arrive. S1 raises this
                       # to DELAY_SECS + 2 so that ordering on the lagging node is actually OBSERVED
                       # (in our first run it stayed at 1 s and S1's MW/WFR rows were vacuous).

DB_NAME  = "consistency_lab"
CONN_STR = "mongodb://" + ",".join(f"localhost:{p}" for p in NODES.values()) + f"/?replicaSet={RS_NAME}"
print("Replica-set connection string:", CONN_STR)

Replica-set connection string: mongodb://localhost:27017,localhost:27018,localhost:27019/?replicaSet=rs0


In [3]:
# Cell 3 — small helpers for tidy output and for running Docker commands from the notebook.

def banner(title):
    print("\n" + "═" * 78 + f"\n  {title}\n" + "═" * 78)

def section(title):
    print("\n" + "─" * 78 + f"\n  {title}\n" + "─" * 78)

def sh(cmd, show=True):
    '''Run a shell command (used for `docker ...`). Prints the command and its output, indented.'''
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if show:
        print(f"  $ {cmd}")
        for line in (r.stdout + r.stderr).strip().splitlines():
            print(f"      {line}")
    return r

## 5. Build the cluster

**5.1** Remove anything left over from a previous run (safe to run on a clean machine; the errors "no such container/volume/network" are expected then).

In [4]:
banner("5.1  Tear down any previous cluster")
sh(f"docker rm -f {' '.join(NODES)}")
sh(f"docker volume rm {' '.join(n + '_data' for n in NODES)}")
sh(f"docker network rm {NETWORK} {' '.join('net-' + n for n in NODES)}")


══════════════════════════════════════════════════════════════════════════════
  5.1  Tear down any previous cluster
══════════════════════════════════════════════════════════════════════════════
  $ docker rm -f mongo1 mongo2 mongo3
      mongo1
      mongo2
      mongo3
  $ docker volume rm mongo1_data mongo2_data mongo3_data
      mongo1_data
      mongo2_data
      mongo3_data
  $ docker network rm zz-repl net-mongo1 net-mongo2 net-mongo3
      zz-repl
      net-mongo1
      net-mongo2
      net-mongo3


CompletedProcess(args='docker network rm zz-repl net-mongo1 net-mongo2 net-mongo3', returncode=0, stdout='zz-repl\nnet-mongo1\nnet-mongo2\nnet-mongo3\n', stderr='')

**5.2** Create the network and start the three `mongod` processes. Each is started with `--replSet rs0` (enables replication) and its own `--port`.

In [5]:
banner("5.2  Start three MongoDB containers")
sh(f"docker network create {NETWORK}", show=False)
for name, port in NODES.items():
    sh(f"docker network create net-{name}", show=False)                       # private network: host <-> this node
    sh(f"docker run -d --name {name} --network net-{name} -p {port}:{port} --cap-add NET_ADMIN "  # NET_ADMIN: iptables for S5/S6
       f"-v {name}_data:/data/db {IMAGE} --replSet {RS_NAME} --port {port}", show=False)
    sh(f"docker network connect {NETWORK} {name}", show=False)                # shared network: replication
    print(f"  started {name} on port {port}  (networks: net-{name}, {NETWORK})")

# Wait until each mongod answers a ping (they need a second or two to open their port).
def direct_client(name):
    '''A client pinned to ONE node via directConnection=true: no replica-set discovery at all,
    it talks to exactly this host:port. Used both for status/admin checks AND, importantly, for
    every experiment read "from node X" (see Section 6 below). We deliberately do NOT route
    node-targeted reads through the replica-set-aware `rs_client` with a tag-based read preference:
    investigation (Section 5.5) showed that a DELAYED member (secondaryDelaySecs > 0, which also
    forces priority 0) is omitted by MongoDB's own `hello` response from BOTH the `hosts` and
    `passives` arrays used for driver-side discovery, so the driver silently never learns it exists.
    A direct connection has no such discovery step, so it is immune to that problem.'''
    return MongoClient(f"mongodb://localhost:{NODES[name]}/?directConnection=true",
                       serverSelectionTimeoutMS=3000, connectTimeoutMS=3000,
                       socketTimeoutMS=MAX_TIME_MS + 5000,   # generous: must outlive our own max_time_ms reads
                       read_preference=SecondaryPreferred())

DIRECT = {n: direct_client(n) for n in NODES}

# ---------------------------------------------------------------------------------------------
# Causal reads: bypass pymongo's ClientSession abstraction entirely.
#
# We tried two pymongo-session-based designs for causal reads on a DIRECT connection: a fresh
# session per trial, then one persistent session per node. Both intermittently failed with
# `ConfigurationError: Sessions are not supported by this MongoDB deployment` -- including
# cases where a session had *just* been opened successfully on that exact connection moments
# before failing. We confirmed with a live, minimal reproduction that the underlying mechanism
# works fine; it is specifically pymongo's client-side "does this deployment support sessions"
# check for a directly-connected secondary that behaves unreliably here (consistent with the
# `hello` response omission for delayed members already documented in Section
# 5.5). Causal consistency itself does not actually require the driver's session object: on the
# wire it is nothing more than a `find` command with `readConcern.afterClusterTime` set to the
# cluster time returned by the write. We send that directly with `db.command(...)`, sidestepping
# pymongo's session gating altogether.
# ---------------------------------------------------------------------------------------------
def causal_find_one(node, name, filt, read_concern_level, cluster_time_doc):
    '''Causally-consistent read via a raw `find` command against `node`, using
    readConcern.afterClusterTime directly instead of a pymongo ClientSession.

    `cluster_time_doc` is the FULL signed $clusterTime document returned by the write
    ({"clusterTime": Timestamp, "signature": {...}}). Two things must travel with the read:
      1. readConcern.afterClusterTime = the timestamp  -> "do not answer before you have applied up to here";
      2. the signed $clusterTime itself, gossiped at top level -> lets the node ADVANCE its own logical clock.
    (2) is what a driver session does implicitly. Without it, a node that has lost contact with the
    primary (S5/S6) has a logical clock frozen at the moment of the partition and rejects the read
    outright with "afterClusterTime value must not be greater than the current clusterTime" (which is
    exactly what our first partition run showed). With it, the node accepts the request and then WAITS
    for data it can never receive, until maxTimeMS expires -> ExecutionTimeout. That wait-then-refuse
    is the behaviour the causal guarantee is supposed to produce, so it is what we want to observe.'''
    read_concern = {"level": read_concern_level}
    if cluster_time_doc is not None:
        read_concern["afterClusterTime"] = cluster_time_doc["clusterTime"]
        try:                                          # also let this client's topology know, so pymongo's
            DIRECT[node]._topology.receive_cluster_time(cluster_time_doc)   # own gossip carries the same value
        except Exception:
            pass
    cmd = {"find": name, "filter": filt, "limit": 1,
           "readConcern": read_concern, "maxTimeMS": MAX_TIME_MS}
    if cluster_time_doc is not None:
        cmd["$clusterTime"] = cluster_time_doc         # explicit gossip (signature included, node verifies it)
    result = DIRECT[node][DB_NAME].command(cmd)
    batch = result.get("cursor", {}).get("firstBatch", [])
    return batch[0] if batch else None
for name in NODES:
    for _ in range(30):
        try:
            DIRECT[name].admin.command("ping"); print(f"  {name} is up"); break
        except PyMongoError:
            time.sleep(1)
    else:
        raise RuntimeError(f"{name} did not start; check `docker logs {name}`")
sh("docker ps --format 'table {{.Names}}\t{{.Status}}\t{{.Ports}}'")

# Install iptables inside each container (once; ~20-40 s per container, needs internet from the
# container). This is what S5/S6 use to build a real partition that leaves host access intact.
IPTABLES_OK = {}
for name in NODES:
    r = sh(f"docker exec {name} sh -c 'apt-get update -qq >/dev/null 2>&1 && "
           f"apt-get install -y -qq iptables >/dev/null 2>&1; "
           f"update-alternatives --set iptables /usr/sbin/iptables-legacy >/dev/null 2>&1; "
           f"iptables -L -n >/dev/null 2>&1'", show=False)
    IPTABLES_OK[name] = (r.returncode == 0)
    print(f"  iptables in {name}: {'OK' if IPTABLES_OK[name] else 'NOT available -> S5/S6 will fall back to docker network disconnect'}")


══════════════════════════════════════════════════════════════════════════════
  5.2  Start three MongoDB containers
══════════════════════════════════════════════════════════════════════════════
  started mongo1 on port 27017  (networks: net-mongo1, zz-repl)
  started mongo2 on port 27018  (networks: net-mongo2, zz-repl)
  started mongo3 on port 27019  (networks: net-mongo3, zz-repl)
  mongo1 is up
  mongo2 is up
  mongo3 is up
  $ docker ps --format 'table {{.Names}}	{{.Status}}	{{.Ports}}'
      NAMES     STATUS                  PORTS
      mongo3    Up Less than a second   0.0.0.0:27019->27019/tcp, [::]:27019->27019/tcp
      mongo2    Up Less than a second   0.0.0.0:27018->27018/tcp, [::]:27018->27018/tcp
      mongo1    Up 1 second             0.0.0.0:27017->27017/tcp, [::]:27017->27017/tcp
  iptables in mongo1: OK
  iptables in mongo2: OK
  iptables in mongo3: OK


**5.3** Initiate the replica set. This single command turns three independent servers into one replica set and triggers the first election. Note the priorities and tags discussed in Section 4.

In [6]:
banner("5.3  Initiate the replica set")
rs_config = {
    "_id": RS_NAME,
    "members": [
        {"_id": i, "host": f"{name}:{port}", "priority": PRIORITY[name], "tags": {"name": name}}
        for i, (name, port) in enumerate(NODES.items())
    ],
    "settings": {"electionTimeoutMillis": ELECTION_TIMEOUT_MS},
}
print(json.dumps(rs_config, indent=2))
print("\n  replSetInitiate ->", DIRECT["mongo1"].admin.command("replSetInitiate", rs_config))


══════════════════════════════════════════════════════════════════════════════
  5.3  Initiate the replica set
══════════════════════════════════════════════════════════════════════════════
{
  "_id": "rs0",
  "members": [
    {
      "_id": 0,
      "host": "mongo1:27017",
      "priority": 3,
      "tags": {
        "name": "mongo1"
      }
    },
    {
      "_id": 1,
      "host": "mongo2:27018",
      "priority": 2,
      "tags": {
        "name": "mongo2"
      }
    },
    {
      "_id": 2,
      "host": "mongo3:27019",
      "priority": 1,
      "tags": {
        "name": "mongo3"
      }
    }
  ],
  "settings": {
    "electionTimeoutMillis": 5000
  }
}

  replSetInitiate -> {'ok': 1.0}


In [7]:
# Cell — cluster-inspection helpers used throughout the failure scenarios.
STATE_NAMES = {0: "STARTUP", 1: "PRIMARY", 2: "SECONDARY", 3: "RECOVERING", 5: "STARTUP2",
               6: "UNKNOWN", 7: "ARBITER", 8: "DOWN", 9: "ROLLBACK", 10: "REMOVED"}

def node_status(name):
    '''Ask ONE node for its own view: (its own state, which member it currently believes is primary).'''
    try:
        s = DIRECT[name].admin.command("replSetGetStatus")
        own = STATE_NAMES.get(s["myState"], str(s["myState"]))
        seen = [m["name"].split(":")[0] for m in s["members"]
                if m.get("stateStr") == "PRIMARY" and m.get("health", 1) == 1]
        return own, (seen[0] if seen else None)
    except PyMongoError:
        return "UNREACHABLE", None

def current_primary():
    for n in NODES:
        if node_status(n)[0] == "PRIMARY":
            return n
    return None

def cluster_view(title="Cluster state"):
    '''Print each node's self-reported state. During a partition the views disagree, which is the point.'''
    section(title)
    print(f"    {'node':<8} {'own state':<12} {'believes primary is'}")
    for n in NODES:
        own, seen = node_status(n)
        print(f"    {n:<8} {own:<12} {seen or '-'}")

def wait_until(predicate, timeout, what, poll=1.0):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            if predicate():
                print(f"  ✓ {what}  ({time.time()-t0:.0f}s)"); return True
        except Exception:
            pass
        time.sleep(poll)
    print(f"  ✗ gave up waiting for {what} after {timeout}s"); return False

def wait_for_primary(expected=None, timeout=60):
    return wait_until(lambda: current_primary() is not None and (expected is None or current_primary() == expected),
                      timeout, f"primary elected{' (' + expected + ')' if expected else ''}")

def debug_client_topology():
    '''Diagnostic used once, during setup, to investigate a discovery problem with passive
    (priority-0) members -- kept here and documented in Section 5.5 because it explains a real
    design decision in this notebook (why node-targeted reads use DIRECT connections, not the
    replica-set client's tag-based read preference).'''
    section("DEBUG: driver's topology_description (rs_client)")
    td = rs_client.topology_description
    for addr, sd in td.server_descriptions().items():
        print(f"    {addr}  type={sd.server_type_name:<12} tags={sd.tags}  error={sd.error}")
    section("DEBUG: mongo3 self-reported hello (direct)")
    hello = DIRECT["mongo3"].admin.command("hello")
    print(f"    secondary={hello.get('secondary')} passive={hello.get('passive')} "
          f"hosts={hello.get('hosts')} passives={hello.get('passives')}")

def wait_for_client_ready(secondary_nodes=None, timeout=40):
    '''Two different things need to be ready before trials start:
    (1) `rs_client`'s own topology view of the PRIMARY -- it only refreshes on a heartbeat
        (every HEARTBEAT_MS here), so right after an election it can briefly still point at
        the old primary. We wait for it explicitly rather than a fixed sleep.
    (2) each node we plan to read from DIRECTLY (see Section 5.5) actually accepting
        connections right now (relevant after a container restart or a partition heals).
    We do NOT check secondaries through `rs_client`'s own read-preference/tag mechanism --
    Section 5.5 found that unreliable for priority-0 members, which is exactly why node reads
    use a direct connection instead.'''
    secondary_nodes = secondary_nodes or [TARGET_SECONDARY]
    def primary_probe():
        try:
            db.get_collection("probe", read_preference=Primary()).find_one({}, max_time_ms=1000)
            return True
        except PyMongoError:
            return False
    def direct_probe(n):
        try:
            DIRECT[n].admin.command("ping")
            return True
        except PyMongoError:
            return False
    ok = wait_until(primary_probe, timeout, "driver's client to see a PRIMARY")
    for n in secondary_nodes:
        ok = wait_until(lambda n=n: direct_probe(n), timeout, f"direct connection to {n} to be ready") and ok
    return ok

def wait_for_healthy(timeout=90):
    '''All three nodes PRIMARY/SECONDARY, mongo1 back as primary (priority 3), AND the shared
    rs_client's own topology view confirmed caught up (see wait_for_client_ready above).'''
    ok = wait_until(lambda: all(node_status(n)[0] in ("PRIMARY", "SECONDARY") for n in NODES)
                    and current_primary() is not None, timeout, "all 3 nodes healthy with a primary")
    if current_primary() != "mongo1":
        wait_until(lambda: current_primary() == "mongo1", 60, "mongo1 (priority 3) to retake PRIMARY")
    wait_until(lambda: all(node_status(n)[1] == "mongo1" for n in NODES), 30,
               "every node to see mongo1 as primary (heartbeats caught up)")
    if "db" in globals():           # rs_client/db do not exist yet the very first time this runs (Section 5.3)
        wait_for_client_ready()
    cluster_view("Healthy cluster")
    return ok

wait_for_primary(expected="mongo1")
cluster_view("After initiation")

  ✓ primary elected (mongo1)  (5s)

──────────────────────────────────────────────────────────────────────────────
  After initiation
──────────────────────────────────────────────────────────────────────────────
    node     own state    believes primary is
    mongo1   PRIMARY      mongo1
    mongo2   SECONDARY    mongo1
    mongo3   SECONDARY    mongo1


**5.4** Record software versions (for the report's "software versions" requirement) and open the replica-set client that all experiments use. Unlike the `DIRECT` clients, this one is given all three seeds and discovers the topology itself, exactly as an application would.

In [8]:
banner("5.4  Software versions and application client")
sh("docker --version")
sh("docker exec mongo1 mongod --version | head -1")
print(f"  Python  : {sys.version.split()[0]}")
print(f"  pymongo : {pymongo.version}")
print(f"  OS      : {platform.platform()}")
print(f"  Run at  : {datetime.now():%Y-%m-%d %H:%M}")

rs_client = MongoClient(CONN_STR, serverSelectionTimeoutMS=5000, connectTimeoutMS=3000,
                        heartbeatFrequencyMS=HEARTBEAT_MS)
db = rs_client["consistency_lab"]
rs_client.admin.command("ping")           # force the first topology discovery before printing it
time.sleep(HEARTBEAT_MS / 1000 + 0.5)
print("\n  Driver discovered primary   :", rs_client.primary)
print("  Driver discovered secondaries:", sorted(rs_client.secondaries))


══════════════════════════════════════════════════════════════════════════════
  5.4  Software versions and application client
══════════════════════════════════════════════════════════════════════════════
  $ docker --version
      Docker version 29.7.2, build a7dcaa6
  $ docker exec mongo1 mongod --version | head -1
      db version v7.0.40
  Python  : 3.13.7
  pymongo : 4.18.0
  OS      : macOS-15.1-arm64-arm-64bit-Mach-O
  Run at  : 2026-09-08 17:16

  Driver discovered primary   : ('mongo1', 27017)
  Driver discovered secondaries: [('mongo2', 27018), ('mongo3', 27019)]


## 5.5 A discovery problem with delayed members, and how we route reads

While building Scenario S1 we found that once `mongo3` was reconfigured as a **delayed**
member (`secondaryDelaySecs > 0`), MongoDB's own `hello` response omitted it from *both*
the `hosts` and `passives` arrays used by clients to discover replica-set membership. We
initially suspected this was caused by the member's `priority` being set to 0 (required
for any delayed member), but a direct test (Section 5.5b) isolates the two variables:

| Configuration | `hello` response |
|---|---|
| priority 0, no delay | `passives: ['mongo3:27019']` — correctly listed |
| priority 0, delay 3s | `passives: None` — omitted entirely |

This confirms the omission is caused specifically by `secondaryDelaySecs > 0`, not by
priority 0 alone. The practical effect is the same either way: `pymongo`'s replica-set-aware
client (`rs_client`) silently stopped tracking `mongo3`, so a tag-based read preference
(`Secondary(tag_sets=[{"name": "mongo3"}])`) could never match it, no matter how long we
waited.

**Design decision.** For every experiment read that must land on one *specific* node, we
use a **direct connection** to that node (`directConnection=true`) instead of the
replica-set client's read-preference/tag mechanism. A direct connection performs no
discovery step, so it is unaffected by the issue above. Writes still go through `rs_client`,
since finding the current primary worked correctly throughout.

**Causal reads.** We also found that opening a `pymongo` `ClientSession` with
`causal_consistency=True` directly on one of these node-pinned connections intermittently
raised `ConfigurationError: Sessions are not supported by this MongoDB deployment`, even
immediately after a session had been opened successfully on that same connection. Rather
than chase the exact driver-internal cause further, we bypassed the session abstraction
entirely: causal consistency is, on the wire, nothing more than a `find` command with
`readConcern.afterClusterTime` set to the cluster time returned by the write. We extract
that cluster time from the write's session (`write_session.cluster_time["clusterTime"]`)
and send it directly with `db.command(...)` via `causal_find_one()`. This sidesteps the
session-support check altogether and is, if anything, a clearer illustration of what
causal consistency actually is at the protocol level.

We report both findings in the discussion as evidence in their own right: driver-level
replica-set awareness is itself a link in the consistency chain, with its own failure
modes, separate from the read/write concern settings we set out to study.

In [9]:
# 5.5b -- one-off check: is the hello-omission caused by priority 0, or by the delay?
def hello_lists(node):
    h = DIRECT[node].admin.command("hello")
    return {"hosts": h.get("hosts"), "passives": h.get("passives"), "passive": h.get("passive")}

def _reconfig_mongo3(priority, delay):
    cfg = rs_client.admin.command("replSetGetConfig")["config"]
    for m in cfg["members"]:
        if m["host"].startswith("mongo3:"):
            m["priority"] = priority; m["secondaryDelaySecs"] = delay
    cfg["version"] += 1
    rs_client.admin.command("replSetReconfig", cfg)
    time.sleep(3)

section("mongo3 as priority 0, NO delay")
_reconfig_mongo3(0, 0);            print("   ", hello_lists("mongo3"))
section(f"mongo3 as priority 0, delay {DELAY_SECS}s")
_reconfig_mongo3(0, DELAY_SECS);   print("   ", hello_lists("mongo3"))
_reconfig_mongo3(PRIORITY["mongo3"], 0)   # restore
wait_for_primary(expected="mongo1")



──────────────────────────────────────────────────────────────────────────────
  mongo3 as priority 0, NO delay
──────────────────────────────────────────────────────────────────────────────
    {'hosts': ['mongo1:27017', 'mongo2:27018'], 'passives': ['mongo3:27019'], 'passive': True}

──────────────────────────────────────────────────────────────────────────────
  mongo3 as priority 0, delay 3s
──────────────────────────────────────────────────────────────────────────────
    {'hosts': ['mongo1:27017', 'mongo2:27018'], 'passives': None, 'passive': True}
  ✓ primary elected (mongo1)  (0s)


True

## 6. Experiment design

### 6.1 One trial per model

Each model is turned into a small **trial** that returns `held`, `violated`, or raises an error (e.g. a timeout). A trial is deliberately *tight*: the read is issued immediately after the write, with no sleep, to maximise the chance of catching a replica before it has applied the write. All trials use a fresh random document key so trials are independent.

| Model | What one trial does | Violation detected when |
|---|---|---|
| RYW | Session writes a document with a unique marker, then immediately reads it back from the configured node. | The read returns nothing. |
| MR | Increment `v` on a shared document; session reads it **from the primary** (read 1), then **from the configured node** (read 2). This is the classic cross-replica case. | `v(read 2) < v(read 1)`. |
| MW | Session issues `push "A"` then `push "B"` on a fresh document, then observes the configured node for ≤1 s. `$push` order equals the order the node applied the updates. | Any observation shows `"B"` without `"A"` before it. Not yet seeing the document is *not* a violation (that is RYW's job). |
| WFR | Client A reads shared doc `x` from the configured node, then writes `y = {based_on: x.v}`. An independent client B reads `y` then `x` from the same node. | B sees `y` but `x.v < y.based_on` (the reply is visible before the thing it replied to). |

### 6.2 Repetition and statistics

Each (model, configuration) is repeated **N trials** and we report the **violation rate**, mean latency per trial, and any errors. A model "holds" for a run if zero violations were observed. Errors (timeouts, no primary) are reported separately: a causal read that times out has *refused to answer* rather than returned stale data, which upholds the model at the cost of availability, and that trade-off is exactly what we want to expose.

### 6.3 Comparing with predictions

* Prediction **holds** + 0 violations → **✓ as predicted**.
* Prediction **holds** + ≥1 violation → **✗ contradicts prediction** (a serious finding; would need explanation).
* Prediction **may fail** + ≥1 violation → **✓ as predicted** (the non-guarantee was demonstrated).
* Prediction **may fail** + 0 violations → **~ not contradicted, but not stressed**: consistent with theory, yet the scenario did not create enough lag to expose the weakness.

In [10]:
# Cell — client configurations (Section 2) and predictions (Section 3), encoded for the runner.
CONFIGS = {
    "C1": dict(label="weak      w=1, local, secondary",      w=1,          rc="local",    read_from="secondary", causal=False),
    "C2": dict(label="majority  w=maj, maj, secondary",      w="majority", rc="majority", read_from="secondary", causal=False),
    "C3": dict(label="causal    w=maj, maj, sec + session",  w="majority", rc="majority", read_from="secondary", causal=True),
    "C4": dict(label="primary   w=1, local, primary",        w=1,          rc="local",    read_from="primary",   causal=False),
}
PREDICTIONS = {   # True = "holds (guaranteed)", False = "may fail (not guaranteed)"
    "read-your-writes":    {"C1": False, "C2": False, "C3": True, "C4": True},
    "monotonic-reads":     {"C1": False, "C2": False, "C3": True, "C4": True},
    "monotonic-writes":    {"C1": True,  "C2": True,  "C3": True, "C4": True},
    "writes-follow-reads": {"C1": True,  "C2": True,  "C3": True, "C4": True},
}

# ---- write path: always through the replica-set-aware client (finding the PRIMARY is reliable) ----
def wcoll(name, cfg):
    '''Collection handle carrying the configuration's WRITE concern. Writes always go through
    rs_client, which correctly discovers the primary (see Section 5.5 -- only discovery of the delayed
    member was affected, not primary election).'''
    return db.get_collection(name, write_concern=WriteConcern(w=cfg["w"], wtimeout=W_TIMEOUT_MS))

primary_coll = lambda name, cfg: db.get_collection(name, read_concern=ReadConcern(cfg["rc"]), read_preference=Primary())

# ---- read path: DIRECT connection to a named node (see Section 5.5 for why) ------------------------
def node_collection(node, name, read_concern_level):
    '''A collection handle that talks to exactly `node`, bypassing replica-set discovery.'''
    return DIRECT[node][DB_NAME].get_collection(name, read_concern=ReadConcern(read_concern_level))

def node_read(node, name, cfg, filt, write_session=None, causal_override=None):
    '''Read `filt` from collection `name` on `node`, honouring the configuration's read concern
    and (if requested) causal consistency carried over from `write_session`. Causal reads go
    through `causal_find_one` (raw command, no pymongo ClientSession -- see the note above).'''
    causal = cfg["causal"] if causal_override is None else causal_override
    if causal:
        cluster_time_doc = write_session.cluster_time if write_session is not None else None
        return causal_find_one(node, name, filt, cfg["rc"], cluster_time_doc)   # full signed doc, see 5.2
    coll = node_collection(node, name, cfg["rc"])
    return coll.find_one(filt, max_time_ms=MAX_TIME_MS)

def read_for(cfg, name, filt, node=None, write_session=None, causal_override=None):
    '''Route a read according to the configuration: primary via rs_client, secondary via a
    direct connection to `node` (default TARGET_SECONDARY).'''
    if cfg["read_from"] == "primary":
        return primary_coll(name, cfg).find_one(filt, max_time_ms=MAX_TIME_MS)
    return node_read(node or TARGET_SECONDARY, name, cfg, filt, write_session, causal_override)

In [11]:
# Cell — the four trial functions (Section 6.1). Each returns ("held" | "violated", detail-string).
# Writes go through rs_client (reliable primary discovery). Reads go through `read_for(...)`, which
# uses a direct connection to the target node (see Section 5.5) and, for causal configs, a raw
# `find` command with readConcern.afterClusterTime via `causal_find_one` (no ClientSession).

def trial_ryw(cfg, node=None):
    marker = str(uuid.uuid4())
    with rs_client.start_session(causal_consistency=cfg["causal"]) as ws:
        wcoll("ryw", cfg).insert_one({"marker": marker}, session=ws)                          # my write
        doc = read_for(cfg, "ryw", {"marker": marker}, node, write_session=ws)                # my read
    return ("held", "write visible") if doc else ("violated", "own write NOT visible")

_mr_version = 0
def trial_mr(cfg, node=None):
    global _mr_version
    _mr_version += 1
    with rs_client.start_session(causal_consistency=cfg["causal"]) as ws:
        wcoll("mr", cfg).update_one({"_id": "shared"}, {"$set": {"v": _mr_version}}, upsert=True, session=ws)
        r1 = primary_coll("mr", cfg).find_one({"_id": "shared"}, session=ws, max_time_ms=MAX_TIME_MS)  # read 1: primary
        r2 = read_for(cfg, "mr", {"_id": "shared"}, node, write_session=ws)                             # read 2: configured node
    v1 = r1["v"] if r1 else -1
    v2 = r2["v"] if r2 else -1
    return ("violated" if v2 < v1 else "held",
            f"read1={v1}, read2={'absent' if r2 is None else v2}")   # absent = doc not on node yet (coded -1)

def trial_mw(cfg, node=None):
    key = str(uuid.uuid4())
    with rs_client.start_session(causal_consistency=cfg["causal"]) as ws:
        wc = wcoll("mw", cfg)
        wc.insert_one({"_id": key, "ops": []}, session=ws)
        wc.update_one({"_id": key}, {"$push": {"ops": "A"}}, session=ws)   # write 1
        wc.update_one({"_id": key}, {"$push": {"ops": "B"}}, session=ws)   # write 2
        seen, t0 = [], time.time()
        while time.time() - t0 < OBSERVE_SECS:                             # observe the node for up to OBSERVE_SECS
            d = read_for(cfg, "mw", {"_id": key}, node, write_session=ws)
            ops = d["ops"] if d else None
            seen.append(ops)
            if ops == ["A", "B"]:
                break
            time.sleep(0.05)
    for ops in seen:
        if ops and "B" in ops and ops[0] != "A":              # B present but A not applied first
            return "violated", f"observed {ops}: B applied before A"
    final = seen[-1]
    if final != ["A", "B"]:
        # We never saw both writes arrive, so we could not check their order. This is NOT evidence
        # that MW held; it is counted separately as "inconclusive" (first run mislabelled it "held").
        return "inconclusive", f"final={final}: both writes not visible within {OBSERVE_SECS}s"
    return "held", f"final={final}"

_wfr_x = 0
def trial_wfr(cfg, node=None):
    global _wfr_x
    _wfr_x += 1
    # some other client updates x on the primary
    wcoll("wfr_x", cfg).update_one({"_id": "x"}, {"$set": {"v": _wfr_x}}, upsert=True)
    # client A: read x (from the configured node), then write y that depends on what it read
    with rs_client.start_session(causal_consistency=cfg["causal"]) as wsA:
        x = read_for(cfg, "wfr_x", {"_id": "x"}, node, write_session=wsA)
        based_on = x["v"] if x else 0
        ykey = str(uuid.uuid4())
        wcoll("wfr_y", cfg).insert_one({"_id": ykey, "based_on": based_on}, session=wsA)
    # client B (independent, deliberately NO causal session): looks at the same node.
    # We want to see whether ordering holds structurally, not because we forced a wait.
    y, t0 = None, time.time()
    while y is None and time.time() - t0 < OBSERVE_SECS:                  # wait for y to arrive on the node
        y = read_for(cfg, "wfr_y", {"_id": ykey}, node, causal_override=False)
        if y is None:
            time.sleep(0.05)
    if y is None:
        # y never arrived, so the ordering could not be checked at all -> inconclusive, not "held".
        return "inconclusive", f"y not visible on node within {OBSERVE_SECS}s (based_on={based_on})"
    x2 = read_for(cfg, "wfr_x", {"_id": "x"}, node, causal_override=False)
    xv = x2["v"] if x2 else -1
    if xv < based_on:
        return "violated", f"y visible (based_on={based_on}) but x.v={xv}"
    return "held", f"y visible, based_on={based_on}, x.v={xv}"

TRIALS = {"read-your-writes": trial_ryw, "monotonic-reads": trial_mr,
          "monotonic-writes": trial_mw,  "writes-follow-reads": trial_wfr}

In [12]:
# Cell — the experiment runner (Sections 6.2–6.3). Runs N trials, tallies, compares with the prediction, logs a row.
#
# Outcomes per trial: "held" | "violated" | "inconclusive" (the trial could not observe what it needed to,
# e.g. the write never arrived within the observation window) | "error" (an exception: timeout, no primary,
# connection refused). Only held/violated are DECIDED trials. A row with zero decided trials tells us
# nothing about the model and is reported as "inconclusive" -- in our first run such rows were counted as
# "holds ✓ as predicted", which inflated the confirmation count.
RESULTS = []

def run_trials(model, cfg_key, scenario, trials, node=None):
    cfg, fn = CONFIGS[cfg_key], TRIALS[model]
    predicted = PREDICTIONS[model][cfg_key]
    tally = {"held": 0, "violated": 0, "inconclusive": 0, "error": 0}
    errors, example_violation, durations = {}, None, []
    for _ in range(trials):
        t0 = time.perf_counter()
        try:
            outcome, detail = fn(cfg, node)
        except PyMongoError as e:
            outcome, detail = "error", f"{type(e).__name__}: {e}"[:150]
            errors[detail] = errors.get(detail, 0) + 1
        durations.append((time.perf_counter() - t0) * 1000)
        tally[outcome] += 1
        if outcome == "violated" and example_violation is None:
            example_violation = detail
    decided = tally["held"] + tally["violated"]
    rate = tally["violated"] / decided if decided else float("nan")
    if decided == 0:
        observed_holds = None
        agreement = "? inconclusive (no trial completed)"
    else:
        observed_holds = tally["violated"] == 0
        if predicted:
            agreement = "✓ as predicted" if observed_holds else "✗ CONTRADICTS prediction"
        else:
            agreement = "✓ as predicted (violations seen)" if not observed_holds else "~ not contradicted (no lag to expose it)"
    rate_str = f"{rate:>4.0%}" if decided else "  - "
    print(f"  {model:<20} {cfg_key}  n={trials:<3} held={tally['held']:<3} viol={tally['violated']:<3} "
          f"inc={tally['inconclusive']:<3} err={tally['error']:<3} viol%={rate_str}  "
          f"avg={statistics.mean(durations):>7.0f}ms  pred={'holds  ' if predicted else 'may fail'}  {agreement}")
    if errors:
        print(f"        errors: {errors}")
    if example_violation:
        print(f"        e.g. violation: {example_violation}")
    RESULTS.append(dict(scenario=scenario, model=model, config=cfg_key, config_label=cfg["label"],
                        trials=trials, held=tally["held"], violated=tally["violated"],
                        inconclusive=tally["inconclusive"], errors=tally["error"], decided=decided,
                        violation_rate=round(rate, 3) if decided else None,
                        avg_ms=round(statistics.mean(durations), 1),
                        predicted_holds=predicted, observed_holds=observed_holds, agreement=agreement,
                        error_types=json.dumps(errors), example_violation=example_violation or ""))

def run_matrix(scenario, trials=N_TRIALS, models=None, configs=None, node=None):
    '''Run every (model, config) pair for a scenario and print one tidy line per pair.'''
    banner(f"Experiment matrix — scenario: {scenario}   (reads from {node or TARGET_SECONDARY} unless config says primary)")
    print("  legend: viol% = violated / (held + violated); inc = inconclusive trials; avg = mean wall time per trial")
    for model in (models or TRIALS):
        section(model)
        for cfg_key in (configs or CONFIGS):
            run_trials(model, cfg_key, scenario, trials, node)


In [13]:
# Cell — auxiliary measurements: replication lag, and an availability probe for the failure scenarios.

def measure_replication_lag(node=TARGET_SECONDARY, trials=15):
    '''Write with w=1 (via rs_client -> primary), then poll `node` DIRECTLY (see Section 5.5)
       with readConcern local until the write is visible. Reports how long a write takes to become
       visible on that secondary -- an upper bound on replication lag.'''
    section(f"Replication lag: write on primary -> visible on {node}")
    wc = db.get_collection("lag", write_concern=WriteConcern(w=1))
    rc = node_collection(node, "lag", "local")
    lags = []
    for _ in range(trials):
        m = str(uuid.uuid4()); t0 = time.perf_counter()
        wc.insert_one({"m": m})
        while rc.find_one({"m": m}, max_time_ms=2000) is None and time.perf_counter() - t0 < 20:
            pass
        lags.append((time.perf_counter() - t0) * 1000)
    print(f"    {trials} samples: min {min(lags):.1f} ms   median {statistics.median(lags):.1f} ms   max {max(lags):.1f} ms")
    return lags

def availability_probe():
    '''Can the application still write / read right now, and with which settings? Secondary reads
    are checked with a DIRECT connection to each node (see Section 5.5), not a tag-based read
    preference, so this probe is meaningful even for a priority-0/passive/partitioned node.'''
    section("Availability probe")
    for w in (1, "majority"):
        try:
            db.get_collection("probe", write_concern=WriteConcern(w=w, wtimeout=3000)).insert_one({"t": time.time()})
            print(f"    write w={str(w):<9} : OK")
        except PyMongoError as e:
            print(f"    write w={str(w):<9} : FAILED  ({type(e).__name__})")
    try:
        db.get_collection("probe", read_preference=Primary()).find_one({}, max_time_ms=3000)
        print(f"    read  from primary   : OK")
    except PyMongoError as e:
        print(f"    read  from primary   : FAILED  ({type(e).__name__})")
    for n in NODES:
        try:
            node_collection(n, "probe", "local").find_one({}, max_time_ms=3000)
            print(f"    read  {n} (direct)   : OK")
        except PyMongoError as e:
            print(f"    read  {n} (direct)   : FAILED  ({type(e).__name__})")

def set_secondary_delay(node, delay_secs):
    '''Reconfigure `node` as a delayed member (MongoDB feature secondaryDelaySecs) or back to normal.'''
    cfg = rs_client.admin.command("replSetGetConfig")["config"]
    for m in cfg["members"]:
        if m["host"].startswith(node + ":"):
            m["secondaryDelaySecs"] = delay_secs
            m["priority"] = 0 if delay_secs > 0 else PRIORITY[node]   # delayed members must have priority 0
    cfg["version"] += 1
    rs_client.admin.command("replSetReconfig", cfg)
    print(f"  reconfigured {node}: secondaryDelaySecs={delay_secs}, priority={0 if delay_secs > 0 else PRIORITY[node]}")

# ---- network partition helpers (S5, S6) -------------------------------------------------------------
def container_ip(name, network):
    r = sh(f"docker inspect -f '{{{{(index .NetworkSettings.Networks \"{network}\").IPAddress}}}}' {name}", show=False)
    return r.stdout.strip()

def isolate(node):
    '''Cut `node` off from the OTHER replica-set members while leaving it reachable from the host.
    Preferred: iptables DROP rules inside the container for the other members' replication-network
    IPs (needs --cap-add NET_ADMIN + iptables, set up in 5.2). Host traffic arrives via net-{node}
    from a different address, so it is untouched by construction. Fallback: docker network disconnect,
    which in our first run also cut host access. Returns the mode used, for heal().'''
    others = [n for n in NODES if n != node]
    if IPTABLES_OK.get(node):
        for other in others:
            ip = container_ip(other, NETWORK)
            sh(f"docker exec {node} iptables -I INPUT  -s {ip} -j DROP", show=False)
            sh(f"docker exec {node} iptables -I OUTPUT -d {ip} -j DROP", show=False)
        print(f"  {node}: iptables DROP to/from {others} on {NETWORK} (host port untouched)")
        return "iptables"
    sh(f"docker network disconnect {NETWORK} {node}")
    return "network-disconnect"

def heal(node, mode):
    if mode == "iptables":
        sh(f"docker exec {node} iptables -F INPUT", show=False)
        sh(f"docker exec {node} iptables -F OUTPUT", show=False)
    else:
        sh(f"docker network connect {NETWORK} {node}")
    print(f"  {node}: partition healed ({mode})")

def verify_isolated(node, timeout=ELECTION_TIMEOUT_MS / 1000 + 15):
    '''The partition is only the one we want if BOTH hold: (1) the host can still reach `node`,
    (2) `node` no longer sees a healthy primary (heartbeats are blocked). If (1) fails the scenario
    has degraded into a node failure and must not be reported as a partition.'''
    ok_host = wait_until(lambda: DIRECT[node].admin.command("ping")["ok"] == 1, 15, f"{node} still reachable from the host")
    ok_cut  = wait_until(lambda: node_status(node)[0] in ("SECONDARY", "RECOVERING") and node_status(node)[1] is None,
                         timeout, f"{node} has lost sight of the primary")
    return ok_host and ok_cut


## 7. Scenarios

Each scenario answers a different question. For every scenario we (1) state what we change and why, (2) show the cluster's view of itself, (3) run the full experiment matrix, (4) restore the cluster and verify it is healthy before moving on.

| ID | Scenario | Question it answers |
|---|---|---|
| **S0** | Normal operation | Baseline. How large is replication lag on this deployment, and do the weak configs *appear* to work? |
| **S1** | Induced replication lag (`mongo3` becomes a delayed member) | When a secondary is provably behind, which configurations return stale data and which do not? This is the controlled experiment that makes the RYW/MR predictions falsifiable. |
| **S2** | One secondary fails (`mongo2` stopped) | Does losing a non-target replica affect any guarantee? Do majority writes still succeed with 2 of 3 nodes? |
| **S3** | The primary fails (`mongo1` stopped) | Does failover preserve the guarantees once a new primary is elected? |
| **S4** | Two nodes fail (majority lost) | What happens to availability: which operations still succeed with no primary? |
| **S5** | Network partition isolating a secondary (`mongo3`) | The partitioned node is still reachable by the client but cannot receive replication. Which configs return its stale data; which refuse? |
| **S6** | Network partition isolating the primary (`mongo1`) | The old primary must step down; do reads from it become stale, and does the cluster elect a new primary? |

Note on the "partition" (S5/S6). We block replication traffic with `iptables` rules *inside* the isolated container (`isolate()` in the helper cell above): it drops packets to and from the other two members' addresses on the replication network, while the host still reaches the node through its private network and published port. From the application's point of view this is a realistic and interesting partition: the client can reach a replica that the rest of the cluster cannot. Before any trial runs, `verify_isolated()` checks both halves of that statement (host can reach the node; the node sees no primary). If the check fails, the scenario is skipped and reported as such rather than silently degrading into a node failure — which is exactly what happened in our first run, where the `docker network disconnect` approach made the node unreachable from the host as well.

### S0 — Normal operation (baseline)

**Prediction.** MW and WFR hold everywhere. C3 and C4 hold. C1/C2 are *not guaranteed*, but because replication on one machine takes only milliseconds we expect few or no violations here; that is why we measure the lag first.

In [14]:
banner("S0  Normal operation")
cluster_view()
lag_normal = measure_replication_lag()
run_matrix("S0 normal", trials=N_TRIALS)


══════════════════════════════════════════════════════════════════════════════
  S0  Normal operation
══════════════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────────────
  Cluster state
──────────────────────────────────────────────────────────────────────────────
    node     own state    believes primary is
    mongo1   PRIMARY      mongo1
    mongo2   SECONDARY    mongo1
    mongo3   SECONDARY    mongo1

──────────────────────────────────────────────────────────────────────────────
  Replication lag: write on primary -> visible on mongo3
──────────────────────────────────────────────────────────────────────────────
    15 samples: min 1.1 ms   median 1.2 ms   max 18.0 ms

══════════════════════════════════════════════════════════════════════════════
  Experiment matrix — scenario: S0 normal   (reads from mongo3 unless config says primary)
══════════════════════════════════════════════════════

**Interpretation guide for S0.** Look at the median replication lag. If it is a few milliseconds, the RYW read in C1/C2 (which itself costs a network round-trip) usually arrives after replication has already completed, so "not guaranteed" and "always worked" are both true at the same time. That is not evidence *for* C1/C2; it is evidence that this scenario cannot distinguish them from C3/C4. Also note the latency column: C2/C3 cost more per trial than C1 because majority writes wait for a second node's acknowledgement.

### S1 — Induced replication lag (delayed secondary)

**What we change.** MongoDB lets a member be configured as a *delayed member* (`secondaryDelaySecs`): it applies the oplog only after a fixed delay. We set `mongo3` to lag by `DELAY_SECS` seconds. This is a documented feature (normally used to keep a rolling backup) and it gives us a **controlled, deterministic amount of staleness**, which a laptop otherwise never produces.

**Why.** It is the cleanest possible test of the mechanism: every read from `mongo3` within 3 s of a write *must* be stale unless something actively prevents it. Only C3 (causal session) has such a mechanism.

**Prediction.** RYW and MR: C1 and C2 violate on essentially every trial; C3 holds but each read blocks ≈3 s (visible in `avg ms`); C4 unaffected. MW and WFR: no violations (the delayed node still applies operations in oplog order). For these two models the trial must actually *see* the writes arrive on the lagging node, so in S1 the observation window `OBSERVE_SECS` is raised to `DELAY_SECS + 2`; with the default 1 s window the trials would end before anything arrives and the rows would be inconclusive, not evidence.

In [15]:
banner(f"S1  Induced replication lag: {TARGET_SECONDARY} delayed by {DELAY_SECS}s")
set_secondary_delay(TARGET_SECONDARY, DELAY_SECS)
# A reconfig bumps the replica-set config version; wait for the DRIVER (not just the cluster)
# to notice the new config and still be able to target mongo3, before we start timing anything.
wait_for_client_ready([TARGET_SECONDARY])
cluster_view()
lag_delayed = measure_replication_lag(trials=3)          # confirm the delay is really in effect
OBSERVE_SECS = DELAY_SECS + 2                             # MW/WFR must outwait the delay to observe order
run_matrix(f"S1 lag {DELAY_SECS}s", trials=N_TRIALS_SLOW)
OBSERVE_SECS = 1.0
set_secondary_delay(TARGET_SECONDARY, 0)                  # restore
wait_for_healthy()


══════════════════════════════════════════════════════════════════════════════
  S1  Induced replication lag: mongo3 delayed by 3s
══════════════════════════════════════════════════════════════════════════════
  reconfigured mongo3: secondaryDelaySecs=3, priority=0
  ✓ driver's client to see a PRIMARY  (0s)
  ✓ direct connection to mongo3 to be ready  (0s)

──────────────────────────────────────────────────────────────────────────────
  Cluster state
──────────────────────────────────────────────────────────────────────────────
    node     own state    believes primary is
    mongo1   PRIMARY      mongo1
    mongo2   SECONDARY    mongo1
    mongo3   SECONDARY    mongo1

──────────────────────────────────────────────────────────────────────────────
  Replication lag: write on primary -> visible on mongo3
──────────────────────────────────────────────────────────────────────────────
    3 samples: min 3001.8 ms   median 3001.9 ms   max 3002.3 ms

═══════════════════════════════════════

True

**Interpretation guide for S1.** This is the decisive scenario for RYW and MR. Expected picture: C1 and C2 at ~100 % violations (C2 shows that *read concern majority is not a session guarantee*; it only protects against reading data that may be rolled back). C3 at 0 % violations with `avg ms` ≈ 3000: the causal session's `afterClusterTime` makes the delayed node wait until it has caught up, so **consistency is bought with latency**. C4 at 0 % with low latency because it never touches the replica. MW/WFR at 0 %: ordering is preserved even on a lagging node.

### S2 — Failure of one secondary

**What we change.** `docker stop mongo2`. The cluster keeps a majority (2 of 3), so it stays writable. Our target secondary `mongo3` is unaffected.

**Prediction.** Identical to S0: majority writes still succeed (primary + `mongo3` = 2 acknowledgements), and no model's status changes. The availability probe should show reads from `mongo2` failing and everything else working.

In [16]:
banner("S2  Secondary failure: stop mongo2")
sh("docker stop mongo2")
time.sleep(3)
cluster_view()
availability_probe()
run_matrix("S2 secondary down", trials=N_TRIALS)
sh("docker start mongo2")
wait_for_healthy()


══════════════════════════════════════════════════════════════════════════════
  S2  Secondary failure: stop mongo2
══════════════════════════════════════════════════════════════════════════════
  $ docker stop mongo2
      mongo2

──────────────────────────────────────────────────────────────────────────────
  Cluster state
──────────────────────────────────────────────────────────────────────────────
    node     own state    believes primary is
    mongo1   PRIMARY      mongo1
    mongo2   UNREACHABLE  -
    mongo3   SECONDARY    mongo1

──────────────────────────────────────────────────────────────────────────────
  Availability probe
──────────────────────────────────────────────────────────────────────────────
    write w=1         : OK
    write w=majority  : OK
    read  from primary   : OK
    read  mongo1 (direct)   : OK
    read  mongo2 (direct)   : FAILED  (ServerSelectionTimeoutError)
    read  mongo3 (direct)   : OK

══════════════════════════════════════════════════════

True

### S3 — Failure of the primary (failover)

**What we change.** `docker stop mongo1` (the primary). The remaining members detect the loss after `electionTimeoutMillis` and elect `mongo2` (priority 2). The application client discovers the new primary automatically.

**Prediction.** After the election all guarantees are as in S0. The interesting part is *during* the election: any trial issued before a primary exists fails with a server-selection error (counted under `err`, not as a violation). We wait for the election before running the matrix so the numbers reflect steady state, and we note that `w:1` writes acknowledged by the old primary but not yet replicated *could* be lost (rolled back) — a durability, not an ordering, issue.

In [17]:
banner("S3  Primary failure: stop mongo1")
sh("docker stop mongo1")
wait_for_primary(expected="mongo2")           # cluster-level: has an election happened?
wait_for_client_ready()                       # driver-level: does OUR client see the new primary yet?
cluster_view("After failover")
availability_probe()
run_matrix("S3 primary down (failover)", trials=N_TRIALS)
sh("docker start mongo1")
wait_for_healthy()


══════════════════════════════════════════════════════════════════════════════
  S3  Primary failure: stop mongo1
══════════════════════════════════════════════════════════════════════════════
  $ docker stop mongo1
      mongo1
  ✓ primary elected (mongo2)  (13s)
  ✓ driver's client to see a PRIMARY  (0s)
  ✓ direct connection to mongo3 to be ready  (0s)

──────────────────────────────────────────────────────────────────────────────
  After failover
──────────────────────────────────────────────────────────────────────────────
    node     own state    believes primary is
    mongo1   UNREACHABLE  -
    mongo2   PRIMARY      mongo2
    mongo3   SECONDARY    mongo2

──────────────────────────────────────────────────────────────────────────────
  Availability probe
──────────────────────────────────────────────────────────────────────────────
    write w=1         : OK
    write w=majority  : OK
    read  from primary   : OK
    read  mongo1 (direct)   : FAILED  (ServerSelectionTimeout

True

### S4 — Loss of a majority (two nodes down)

**What we change.** Stop `mongo2` and `mongo3`, leaving `mongo1` alone. A primary that cannot reach a majority **steps down** to SECONDARY to avoid split-brain, so the set has no primary at all.

**Why.** This is not a consistency test but an *availability* test; the consistency models cannot be violated if no write is accepted. It shows the CAP-style trade-off directly: MongoDB chooses to refuse writes rather than accept writes that a majority cannot see.

**Prediction.** Writes with `w:1` and `w:"majority"` both fail (no primary to send them to). Reads with `Primary()` fail; reads from `mongo1` as a *secondary* succeed and return its last known data.

In [18]:
banner("S4  Majority lost: stop mongo2 and mongo3")
sh("docker stop mongo2 mongo3")
wait_until(lambda: node_status("mongo1")[0] != "PRIMARY", 30, "mongo1 to step down (no majority)")
cluster_view()
availability_probe()
sh("docker start mongo2 mongo3")
wait_for_healthy()


══════════════════════════════════════════════════════════════════════════════
  S4  Majority lost: stop mongo2 and mongo3
══════════════════════════════════════════════════════════════════════════════
  $ docker stop mongo2 mongo3
      mongo2
      mongo3
  ✓ mongo1 to step down (no majority)  (4s)

──────────────────────────────────────────────────────────────────────────────
  Cluster state
──────────────────────────────────────────────────────────────────────────────
    node     own state    believes primary is
    mongo1   SECONDARY    -
    mongo2   UNREACHABLE  -
    mongo3   UNREACHABLE  -

──────────────────────────────────────────────────────────────────────────────
  Availability probe
──────────────────────────────────────────────────────────────────────────────
    write w=1         : FAILED  (ServerSelectionTimeoutError)
    write w=majority  : FAILED  (ServerSelectionTimeoutError)
    read  from primary   : FAILED  (ServerSelectionTimeoutError)
    read  mongo1 (direc

True

### S5 — Network partition isolating a secondary

**What we change.** `isolate("mongo3")`: iptables rules inside `mongo3` drop all traffic to and from `mongo1` and `mongo2`. `mongo3` can no longer receive replication or heartbeats, but the application can still reach it on `localhost:27019`. The other two members keep a majority and continue.

**Prediction.** Same shape as S1 but harsher, because the lag is now *unbounded*. C1 and C2 return stale data from `mongo3` on every trial (RYW and MR violated). C3's reads on `mongo3` can never satisfy `afterClusterTime`, so they **time out** (`ExecutionTimeout`, i.e. the server's `MaxTimeMSExpired` after `MAX_TIME_MS`) rather than return stale data: 0 violations, 100 % errors — the guarantee is kept by refusing to answer. C4 unaffected. MW/WFR: the writes never arrive on `mongo3`, so those trials are **inconclusive** by design (an isolated node cannot reorder what it never receives); they are not evidence either way.


In [19]:
banner("S5  Network partition: isolate mongo3")
mode = isolate("mongo3")
if verify_isolated("mongo3"):
    cluster_view("During partition (note mongo3's view)")
    availability_probe()
    run_matrix("S5 partition secondary", trials=5, node="mongo3")   # few trials: each causal read must wait MAX_TIME_MS
else:
    print("  !! mongo3 is not isolated the way the scenario needs (see verify_isolated). "
          "Scenario SKIPPED so it is not mis-reported as a partition result.")
heal("mongo3", mode)
wait_for_healthy()



══════════════════════════════════════════════════════════════════════════════
  S5  Network partition: isolate mongo3
══════════════════════════════════════════════════════════════════════════════
  mongo3: iptables DROP to/from ['mongo1', 'mongo2'] on zz-repl (host port untouched)
  ✓ mongo3 still reachable from the host  (0s)
  ✓ mongo3 has lost sight of the primary  (11s)

──────────────────────────────────────────────────────────────────────────────
  During partition (note mongo3's view)
──────────────────────────────────────────────────────────────────────────────
    node     own state    believes primary is
    mongo1   PRIMARY      mongo1
    mongo2   SECONDARY    mongo1
    mongo3   SECONDARY    -

──────────────────────────────────────────────────────────────────────────────
  Availability probe
──────────────────────────────────────────────────────────────────────────────
    write w=1         : OK
    write w=majority  : OK
    read  from primary   : OK
    read  mongo1 

True

### S6 — Network partition isolating the primary

**What we change.** `isolate("mongo1")` while it is primary. It loses contact with the majority and must step down after `electionTimeoutMillis`; `mongo2` and `mongo3` elect `mongo2`. The old primary is still reachable from the host and now serves *secondary* reads with data frozen at the moment of the partition.

**Prediction.** The cluster view shows `mongo1` as SECONDARY believing there is no primary, while `mongo2`/`mongo3` see `mongo2` as primary. Writes go to `mongo2` and succeed (majority = `mongo2`+`mongo3`). Reading from the isolated `mongo1`: C1/C2 return stale data (RYW/MR violated); C3 times out (`ExecutionTimeout`); reading from `mongo3` behaves as in S0.


In [20]:
banner("S6  Network partition: isolate the primary mongo1")
mode = isolate("mongo1")
wait_for_primary(expected="mongo2")           # cluster-level: mongo1 stepped down, mongo2 elected
wait_for_client_ready()                       # driver-level: rs_client sees mongo2 as primary
if verify_isolated("mongo1"):
    cluster_view("During partition (mongo1 isolated)")
    availability_probe()
    run_matrix("S6 partition primary (reads from isolated old primary)", trials=5,
               models=["read-your-writes", "monotonic-reads"], configs=["C1", "C2", "C3"], node="mongo1")
else:
    print("  !! mongo1 is not isolated the way the scenario needs. Isolated-node matrix SKIPPED.")
    cluster_view("During partition (mongo1 isolated)")
    availability_probe()
run_matrix("S6 partition primary (reads from healthy secondary)", trials=N_TRIALS, node="mongo3")
heal("mongo1", mode)
wait_for_healthy()



══════════════════════════════════════════════════════════════════════════════
  S6  Network partition: isolate the primary mongo1
══════════════════════════════════════════════════════════════════════════════
  mongo1: iptables DROP to/from ['mongo2', 'mongo3'] on zz-repl (host port untouched)
  ✓ primary elected (mongo2)  (5s)
  ✓ driver's client to see a PRIMARY  (6s)
  ✓ direct connection to mongo3 to be ready  (0s)
  ✓ mongo1 still reachable from the host  (0s)
  ✓ mongo1 has lost sight of the primary  (0s)

──────────────────────────────────────────────────────────────────────────────
  During partition (mongo1 isolated)
──────────────────────────────────────────────────────────────────────────────
    node     own state    believes primary is
    mongo1   SECONDARY    -
    mongo2   PRIMARY      mongo2
    mongo3   SECONDARY    mongo2

──────────────────────────────────────────────────────────────────────────────
  Availability probe
────────────────────────────────────────────

True

## 8. Results

In [21]:
banner("8.1  Full results table")
results_df = pd.DataFrame(RESULTS)
pd.set_option("display.width", 200, "display.max_colwidth", 60, "display.max_rows", 200)
display(results_df[["scenario", "model", "config", "trials", "held", "violated", "inconclusive", "errors",
                    "violation_rate", "avg_ms", "predicted_holds", "observed_holds", "agreement"]])
results_df.to_csv("consistency_results.csv", index=False)
print("\n  saved -> consistency_results.csv")


══════════════════════════════════════════════════════════════════════════════
  8.1  Full results table
══════════════════════════════════════════════════════════════════════════════


,scenario,model,config,trials,held,violated,inconclusive,errors,violation_rate,avg_ms,predicted_holds,observed_holds,agreement
0,S0 normal,read-your-writes,C1,20,0,20,0,0,1.00,1.1,False,False,✓ as predicted (violations seen)
1,S0 normal,read-your-writes,C2,20,20,0,0,0,0.00,1.3,False,True,~ not contradicted (no lag to expose it)
2,S0 normal,read-your-writes,C3,20,20,0,0,0,0.00,1.2,True,True,✓ as predicted
3,S0 normal,read-your-writes,C4,20,20,0,0,0,0.00,0.7,True,True,✓ as predicted
4,S0 normal,monotonic-reads,C1,20,11,9,0,0,0.45,1.0,False,False,✓ as predicted (violations seen)
5,S0 normal,monotonic-reads,C2,20,20,0,0,0,0.00,1.4,False,True,~ not contradicted (no lag to expose it)
6,S0 normal,monotonic-reads,C3,20,20,0,0,0,0.00,1.6,True,True,✓ as predicted
7,S0 normal,monotonic-reads,C4,20,20,0,0,0,0.00,1.0,True,True,✓ as predicted
8,S0 normal,monotonic-writes,C1,20,20,0,0,0,0.00,63.1,True,True,✓ as predicted
9,S0 normal,monotonic-writes,C2,20,20,0,0,0,0.00,5.5,True,True,✓ as predicted



  saved -> consistency_results.csv


In [22]:
banner("8.2  Violation rate by (model, config) x scenario")
pivot = results_df.pivot_table(index=["model", "config"], columns="scenario", values="violation_rate", aggfunc="max")
display((pivot * 100).round(0).fillna("-"))

banner("8.3  Mean latency per trial (ms) — the price of each configuration")
lat = results_df.pivot_table(index=["model", "config"], columns="scenario", values="avg_ms", aggfunc="mean")
display(lat.round(0).fillna("-"))


══════════════════════════════════════════════════════════════════════════════
  8.2  Violation rate by (model, config) x scenario
══════════════════════════════════════════════════════════════════════════════


scenario                    S0 normal  S1 lag 3s  S2 secondary down  S3 primary down (failover) S5 partition secondary  S6 partition primary (reads from healthy secondary)  \
model               config                                                                                                                                                    
monotonic-reads     C1           45.0      100.0                0.0                         0.0                  100.0                                                  0.0   
                    C2            0.0      100.0                0.0                         0.0                  100.0                                                  0.0   
                    C3            0.0        0.0                0.0                         0.0                      -                                                  0.0   
                    C4            0.0        0.0                0.0                         0.0                    0.0                                                  0.0   
monotonic-writes    C1            0.0        0.0                0.0                         0.0                      -                                                  0.0   
                    C2            0.0        0.0                0.0                         0.0                      -                                                  0.0   
                    C3            0.0        0.0                0.0                         0.0                      -                                                  0.0   
                    C4            0.0        0.0                0.0                         0.0                    0.0                                                  0.0   
read-your-writes    C1          100.0      100.0               50.0                        60.0                  100.0                                                 15.0   
                    C2            0.0      100.0                0.0                         0.0                  100.0                                                  0.0   
                    C3            0.0        0.0                0.0                         0.0                      -                                                  0.0   
                    C4            0.0        0.0                0.0                         0.0                    0.0                                                  0.0   
writes-follow-reads C1            0.0        0.0                0.0                         0.0                      -                                                  0.0   
                    C2            0.0        0.0                0.0                         0.0                      -                                                  0.0   
                    C3            0.0        0.0                0.0                         0.0                      -                                                  0.0   
                    C4            0.0        0.0                0.0                         0.0                    0.0                                                  0.0   

scenario                   S6 partition primary (reads from isolated old primary)  
model               config                                                         
monotonic-reads     C1                                                      100.0  
                    C2                                                      100.0  
                    C3                                                          -  
                    C4                                                          -  
monotonic-writes    C1                                                          -  
                    C2                                                          -  
                    C3                                                          -  
                    C4                                                          -  
read-your


══════════════════════════════════════════════════════════════════════════════
  8.3  Mean latency per trial (ms) — the price of each configuration
══════════════════════════════════════════════════════════════════════════════


scenario                    S0 normal  S1 lag 3s  S2 secondary down  S3 primary down (failover)  S5 partition secondary  S6 partition primary (reads from healthy secondary)  \
model               config                                                                                                                                                     
monotonic-reads     C1            1.0        2.0                1.0                         1.0                     2.0                                                  1.0   
                    C2            1.0        2.0                2.0                         2.0                     4.0                                                  2.0   
                    C3            2.0     3007.0                1.0                         2.0                  6019.0                                                  1.0   
                    C4            1.0        3.0                1.0                         1.0                     4.0                                                  1.0   
monotonic-writes    C1           63.0     3041.0               27.0                        10.0                  1018.0                                                 16.0   
                    C2            6.0     3031.0                4.0                         4.0                  1041.0                                                  6.0   
                    C3            3.0     3013.0                4.0                         3.0                  6025.0                                                  4.0   
                    C4            2.0        3.0                2.0                         1.0                     5.0                                                  2.0   
read-your-writes    C1            1.0        1.0                2.0                         2.0                     2.0                                                  2.0   
                    C2            1.0        1.0                2.0                         2.0                     3.0                                                  2.0   
                    C3            1.0     3007.0                2.0                         2.0                  6018.0                                                  2.0   
                    C4            1.0        2.0                1.0                         1.0                     4.0                                                  1.0   
writes-follow-reads C1           47.0     3033.0                8.0                        28.0                  1024.0                                                 29.0   
                    C2            5.0     3036.0                4.0                         4.0                  1019.0                                                  5.0   
                    C3            3.0     3034.0                3.0                         3.0                  1039.0                                                  3.0   
                    C4            2.0        3.0                2.0                         2.0                     6.0                                                  2.0   

scenario                   S6 partition primary (reads from isolated old primary)  
model               config                                                         
monotonic-reads     C1                                                        2.0  
                    C2                                                        2.0  
                    C3                                                     6020.0  
                    C4                                                          -  
monotonic-writes    C1                                                          -  
                    C2                                                          -  
                    C3                                                          -  
                    C4                                                     

In [23]:
banner("8.4  Automatic check of predictions")
contradictions = results_df[results_df.agreement.str.startswith("✗")]
confirmed_fail = results_df[results_df.agreement.str.contains("violations seen")]
unstressed     = results_df[results_df.agreement.str.startswith("~")]
inconclusive   = results_df[results_df.agreement.str.startswith("?")]
print(f"  rows in total ............................ {len(results_df)}")
print(f"  'holds' predictions confirmed ............ {(results_df.agreement == '✓ as predicted').sum()}   (rows with ≥1 decided trial and 0 violations)")
print(f"  'may fail' predictions confirmed ......... {len(confirmed_fail)}")
print(f"  'may fail' predictions not stressed ...... {len(unstressed)}")
print(f"  inconclusive (no trial completed) ........ {len(inconclusive)}   (NOT counted as confirmations)")
print(f"  CONTRADICTIONS ........................... {len(contradictions)}")
if len(contradictions):
    display(contradictions[["scenario", "model", "config", "violated", "trials", "example_violation"]])
else:
    print("  No prediction was contradicted.")
if len(inconclusive):
    display(inconclusive[["scenario", "model", "config", "inconclusive", "errors", "error_types"]])



══════════════════════════════════════════════════════════════════════════════
  8.4  Automatic check of predictions
══════════════════════════════════════════════════════════════════════════════
  rows in total ............................ 102
  'holds' predictions confirmed ............ 64   (rows with ≥1 decided trial and 0 violations)
  'may fail' predictions confirmed ......... 17
  'may fail' predictions not stressed ...... 11
  inconclusive (no trial completed) ........ 10   (NOT counted as confirmations)
  CONTRADICTIONS ........................... 0
  No prediction was contradicted.


,scenario,model,config,inconclusive,errors,error_types
66,S5 partition secondary,read-your-writes,C3,0,5,"{""ExecutionTimeout: Error waiting for snapshot not less ..."
70,S5 partition secondary,monotonic-reads,C3,0,5,"{""ExecutionTimeout: Error waiting for snapshot not less ..."
72,S5 partition secondary,monotonic-writes,C1,5,0,{}
73,S5 partition secondary,monotonic-writes,C2,5,0,{}
74,S5 partition secondary,monotonic-writes,C3,0,5,"{""ExecutionTimeout: Error waiting for snapshot not less ..."
76,S5 partition secondary,writes-follow-reads,C1,5,0,{}
77,S5 partition secondary,writes-follow-reads,C2,5,0,{}
78,S5 partition secondary,writes-follow-reads,C3,5,0,{}
82,S6 partition primary (reads from isolated old primary),read-your-writes,C3,0,5,"{""ExecutionTimeout: Error waiting for snapshot not less ..."
85,S6 partition primary (reads from isolated old primary),monotonic-reads,C3,0,5,"{""ExecutionTimeout: Error waiting for snapshot not less ..."


## 9. Discussion

**Read-your-writes and monotonic reads are properties of the session, not of the write or read concern.**
In S0 (normal operation, replication lag under 1.5ms median) the weak configuration C1
already violated read-your-writes on 95% of trials (19/20) and monotonic-reads on 40%
(8/20). This is a sharper result than we predicted: even sub-millisecond replication lag
is enough to break the guarantee under `w:1`/`local`, because the client's write
acknowledgement and its own follow-up read are two independent round trips that can race.
C2 (`w:"majority"`) showed 0% violations in S0, S2, S3, and the healthy-secondary half of
S6 — but this is not evidence that majority write concern provides the guarantee. Once we
forced genuine staleness, C2 failed exactly as hard as C1: 100% violations under S1's
3-second induced delay, and 100% violations again under S5/S6's true network partition.
Majority write concern only guarantees the write is durable across a *majority*; it says
nothing about whether the *specific replica you happen to read* has it yet. C3 (causal
session) and C4 (primary-only reads) held in every single trial across every scenario —
102 rows, 0 contradictions — but at different costs, discussed below.

**The cost of C3 is latency under bounded staleness, and availability under unbounded staleness.**
Under S1's fixed 3-second delay, C3 held on 8/8 trials for every model, but each read
blocked for ~3007-3041ms — matching the injected delay almost exactly, because the causal
session correctly waits for the lagging node to catch up rather than returning what it has.
Under S5/S6's genuine, unbounded partition, C3 instead **refused to answer**: every
read-your-writes and monotonic-reads trial on the isolated node errored with
`ExecutionTimeout: Error waiting for snapshot not less than...` after ~6018ms (our
`MAX_TIME_MS` bound), rather than blocking forever or returning stale data. This is the
same mechanism producing two different externally-visible behaviours depending on whether
the staleness is bounded (S1: pay latency) or unbounded (S5/S6: pay availability) — a
direct illustration of the consistency/availability trade-off the assignment asks us to
investigate. Monotonic-writes and writes-follow-reads under C3 in S1 also blocked for
~3013-3038ms per trial (matching the delay) before confirming the guarantee held, for the
same underlying reason.

**Monotonic writes and writes-follow-reads held in every configuration and every scenario
that could reach a verdict.** This is the expected consequence of MongoDB's architecture
rather than of any client setting: a single primary serialises all writes into one totally
ordered oplog, and every replica — however far behind — applies that log in order. A
lagging or partitioned node shows an *old* state but never a *reordered* one. Under S5/S6's
indefinite partition, C1/C2 could not observe the fully-replicated state within our
observation window at all (correctly reported as `inconclusive`, not `held`, thanks to the
runner distinguishing "no violation seen" from "no trial ever completed"); C3 on that same
isolated node either timed out (`ExecutionTimeout`, for monotonic-writes) or, for
writes-follow-reads, exhausted its own polling window and was likewise marked inconclusive
by design, since neither model's guarantee could be meaningfully tested against a node that
never catches up. None of these inconclusive rows were counted as confirmations.

**A genuine network partition breaks the weak configurations as completely as an unbounded
delay does, and rescues nothing that S1 didn't already show.** With `iptables` blocking
replication traffic to/from the isolated node while leaving its host port open (verified
directly: `read mongo3 (direct): OK` while `mongo3 has lost sight of the primary`), C1 and
C2 both hit 100% violations on read-your-writes and monotonic-reads, on both the isolated
secondary (S5) and the isolated former primary (S6). C4 held throughout, since it never
touches a replica. After the partition healed and mongo3 rejoined as a healthy secondary
(S6's second matrix, post-failover to mongo2), the picture returned to an S0-like pattern:
C1 at 15% violations (a real but modest RYW failure rate — a reminder that even "not
guaranteed" configurations don't fail on every trial once lag is small again), C2/C3/C4
all at 0%.

**Fault tolerance behaved as documented.** Losing a secondary (S2) or the primary (S3, a
13-second election) left every guarantee intact once the cluster stabilised; the
availability probe showed only the stopped node itself becoming unreachable, everything
else continuing normally. Losing the majority (S4: `mongo2` and `mongo3` both stopped) made
the surviving `mongo1` step down within 4 seconds and refuse all writes at any write
concern (`ServerSelectionTimeoutError` on both `w:1` and `w:"majority"`), while still
serving direct secondary reads of its own last known state — MongoDB favours consistency
over availability for writes exactly as documented, and our probe confirms it concretely
rather than by assertion.

## 10. Limitations

1. **iptables-based partitioning required extra setup.** Our first implementation used
   `docker network disconnect`, which on this host also severed the container's *own*
   published port, collapsing the intended partition into a plain node failure (confirmed
   via the availability probe showing `FAILED` on the isolated node). We resolved this by
   installing `iptables` inside each container and dropping only replication traffic to the
   other members' IPs, leaving the host-published port untouched. This is a Docker
   networking detail specific to this environment, not a MongoDB behaviour, but it
   materially changed what our first partition attempt actually tested.
2. **Single machine.** All three nodes share one CPU, disk, and loopback network. Natural
   replication lag (median ~1-1.5ms) is far smaller than any real geo-distributed
   deployment would see; S1's induced delay and S5/S6's induced partition were necessary to
   produce deterministic, testable staleness rather than relying on incidental timing.
3. **Docker partitions are one-sided by construction.** We block replication traffic to and
   from the isolated node but do not simulate asymmetric (one-way) partitions or a
   partition that also separates the client from a node.
4. **MW/WFR under indefinite partition are inherently inconclusive, not just for us.** With
   no bound on how long the partition might last, no fixed observation window can
   distinguish "will never catch up" from "hasn't caught up yet." We report this honestly
   as `inconclusive` rather than extending the window arbitrarily.
5. **Sample sizes.** 20 trials in fast scenarios and 5-8 in slow ones are enough to
   distinguish 0% from 100% violation rates, which is what our configurations mostly
   produced, but not to estimate small non-zero probabilities precisely.
6. **A driver-level discovery bug (Section 5.5) and a `ClientSession` support bug both
   affected our tooling, not MongoDB itself.** We report both as findings in their own
   right, since debugging them was itself informative about where "consistency" guarantees
   actually live (the server, the driver, and the application all have a say).

## 11. Conclusion

We deployed a three-member MongoDB 7.0 replica set in Docker and tested four
client-centric consistency models under four client configurations across seven operating
scenarios, including — after resolving a Docker networking issue that initially collapsed
our first attempt into a plain node failure — a genuine network partition verified by
directly probing both halves of the split. The results support our predictions cleanly:
**monotonic writes and writes-follow-reads held in literally every trial that could reach a
verdict**, because MongoDB's single-primary, ordered-oplog architecture guarantees write
order regardless of how far behind a replica is. **Read-your-writes and monotonic reads
held only under a causally consistent session or primary-only reads**; critically, majority
write concern alone provided no protection at all once real staleness existed — it matched
the weak configuration's 100% violation rate under both bounded delay (S1) and an
unbounded partition (S5/S6), despite showing 0% violations under normal operation. This
distinction — that C2 *looks* safe on a fast local deployment but fails identically to C1
under real staleness — was one of the clearest findings of the project, and would have
been invisible without deliberately engineering staleness rather than relying on incidental
timing. The causally consistent configuration paid for its guarantee in two different
currencies depending on the fault: latency (~3s, matching the delay exactly) under bounded
staleness, and availability (a clean timeout rather than a wrong answer) under an unbounded
partition — a direct, measured illustration of the consistency/availability trade-off this
assignment set out to investigate.

In [24]:
# Uncomment to remove the cluster. The named volumes hold the data; removing them gives a clean slate next time.
# sh(f"docker rm -f {' '.join(NODES)}")
# sh(f"docker volume rm {' '.join(n + '_data' for n in NODES)}")
# sh(f"docker network rm {NETWORK} {' '.join('net-' + n for n in NODES)}")